# Predicting late delivery
Goal: predict `late_delivery` at dispatch time for future consignments.

Design decisions (per checklist.md):
- **Features:** `claim_paid_eur` is a claim settled after the consignment is delivered, a consequence of lateness, so it is excluded as leakage. `consignment` is an identifier and dropped. Allowed: `carrier`, `distance_km`, `weight_kg`, `parcels`, and hour/weekday from `dispatched_on`.
- **Split:** the model is for future consignments, so the test set is the latest 20% of rows by `dispatched_on`. Each consignment id appears once, so no entity grouping is needed.
- **CV:** expanding-window `TimeSeriesSplit` inside the training rows only; preprocessing lives in a Pipeline so it is fit per fold.

In [ ]:
import pandas as pd, numpy as np
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
assert df.consignment.is_unique
df['hour']=df.dispatched_on.dt.hour
df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape, df.late_delivery.mean())

## 1. Temporal hold-out (set aside before any modelling)

In [ ]:
cut=int(len(df)*0.8)
train,test=df.iloc[:cut].copy(),df.iloc[cut:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
print('train late rate',train.late_delivery.mean(),'test late rate',test.late_delivery.mean())
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']
NUM=['distance_km','weight_kg','parcels','hour','dow']
Xtr,ytr=train[FEATS],train.late_delivery
Xte,yte=test[FEATS],test.late_delivery

## 2. Candidate models (named before any scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: one-hot carrier + standardised numerics, logistic regression (C=1).
3. `hgb_d{3,6}_lr{0.05,0.1}`: histogram gradient boosting, small grid over max_depth and learning rate, `max_iter=150`.

Selection metric: out-of-fold ROC AUC. The decision threshold is then chosen on out-of-fold predictions to maximise F1.

In [ ]:
def pre():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),['carrier']),('n',StandardScaler(),NUM)])
cands={
 'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg':Pipeline([('p',pre()),('m',LogisticRegression(C=1,max_iter=1000))]),
}
for d in (3,6):
    for lr in (0.05,0.1):
        cands[f'hgb_d{d}_lr{lr}']=Pipeline([('p',pre()),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=150,random_state=0))])
print(list(cands))

## 3. Time-ordered cross-validation on training rows only

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
oof={}; rows=[]
for name,p in cands.items():
    pred=np.full(len(Xtr),np.nan); aucs=[]
    for a,b in tscv.split(Xtr):
        m=clone(p).fit(Xtr.iloc[a],ytr.iloc[a])
        pr=m.predict_proba(Xtr.iloc[b])[:,1]; pred[b]=pr
        aucs.append(roc_auc_score(ytr.iloc[b],pr))
    oof[name]=pred
    rows.append((name,np.mean(aucs),np.std(aucs)))
cv=pd.DataFrame(rows,columns=['model','cv_auc_mean','cv_auc_sd']).sort_values('cv_auc_mean',ascending=False)
print(cv)

In [ ]:
best=cv.iloc[0].model
mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.95,91)
f1s=[f1_score(ytr[mask],oof[best][mask]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',thr,'| CV F1:',max(f1s))

## 4. Decision (recorded before touching the test set)
The top model by mean out-of-fold AUC (printed above) with the F1-maximising out-of-fold threshold. Features: carrier, distance, weight, parcels, dispatch hour and weekday. No further tuning after this point.

## 5. Refit on all training rows, score test set once

In [ ]:
final=clone(cands[best]).fit(Xtr,ytr)
p=final.predict_proba(Xte)[:,1]
base=clone(cands['baseline']).fit(Xtr,ytr).predict_proba(Xte)[:,1]
res=pd.DataFrame({'model':[best,'baseline'],
 'test_auc':[roc_auc_score(yte,p),roc_auc_score(yte,base)],
 'test_avg_precision':[average_precision_score(yte,p),average_precision_score(yte,base)],
 'test_brier':[brier_score_loss(yte,p),brier_score_loss(yte,base)]})
print(res)
print('F1 at chosen threshold:',f1_score(yte,p>=thr),'| late rate in test:',yte.mean())

## 6. Conclusion
**Headline (test set, scored once): ROC AUC 0.70 for logistic regression vs 0.50 for the trivial baseline; average precision 0.069 vs a 0.022 prevalence; F1 0.07 at the CV-chosen threshold 0.23.**

- There is some ranking signal from carrier, distance, weight, parcels and dispatch time; CV AUC was about 0.72 and held up on the later hold-out (0.70).
- Practical usefulness is limited. The late rate is not stationary: about 25-30% in 2024, with swings through 2025, and about 1-2% from Oct 2025 on. The test window (latest 20%, from Oct 2025) is in that low-rate regime (2.2% late vs 25.7% in training). The threshold tuned for F1 on the older, high-rate data therefore flags far too many consignments (precision is low), and the Brier score is worse than the prior's because probabilities are miscalibrated for the new base rate.
- Before relying on this model, recalibrate on recent data and find out what changed in late 2025 (carrier operations, promised-date policy, etc.). The test set was not used for any selection.
- `waldean check` flags a low-confidence (0.60) candidate on the threshold; that is a false positive, since the threshold was chosen on out-of-fold training predictions only.